In [1]:
import pickle, os, warnings
import numpy as np
import pandas as pd
import neurokit2 as nk
import xgboost as xgb
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import joblib

warnings.filterwarnings('ignore')


In [2]:
DATA_DIR    = "/Users/username/PycharmProjects/Bitalino/WESAD"   # root folder from output.txt
FS          = 700        # RespiBAN sampling rate (Hz)
WINDOW_SEC  = 60
STEP_SEC    = 30         # 50% overlap
WINDOW      = WINDOW_SEC * FS
STEP        = STEP_SEC   * FS

SUBJECTS = [
    'S2','S3','S4','S5','S6','S7','S8','S9',
    'S10','S11','S13','S14','S15','S16','S17'
]
LABEL_MAP = {1: 0, 2: 1}   # baseline→0, stress→1




In [3]:
def load_subject(sid: str) -> dict:
    path = os.path.join(DATA_DIR, sid, f"{sid}.pkl")
    with open(path, 'rb') as f:
        return pickle.load(f, encoding='latin1')


In [4]:
def ecg_features(ecg: np.ndarray, fs: int) -> dict:
    try:
        signals, _ = nk.ecg_process(ecg, sampling_rate=fs)
        hrv = nk.hrv(signals, sampling_rate=fs, show=False)
        return {
            'hrv_meanNN': hrv['HRV_MeanNN'].iloc[0],
            'hrv_sdnn':   hrv['HRV_SDNN'].iloc[0],
            'hrv_rmssd':  hrv['HRV_RMSSD'].iloc[0],
            'hrv_pnn50':  hrv['HRV_pNN50'].iloc[0],
            'hrv_lf':     hrv['HRV_LF'].iloc[0],
            'hrv_hf':     hrv['HRV_HF'].iloc[0],
            'hrv_lf_hf':  hrv['HRV_LFHF'].iloc[0],
        }
    except Exception:
        return {k: np.nan for k in [
            'hrv_meanNN','hrv_sdnn','hrv_rmssd',
            'hrv_pnn50','hrv_lf','hrv_hf','hrv_lf_hf'
        ]}


def eda_features(eda: np.ndarray, fs: int) -> dict:
    try:
        signals, _ = nk.eda_process(eda, sampling_rate=fs)
        tonic  = signals['EDA_Tonic'].values
        phasic = signals['EDA_Phasic'].values
        peaks  = signals['SCR_Peaks'].values
        return {
            'eda_tonic_mean':  np.mean(tonic),
            'eda_tonic_std':   np.std(tonic),
            'eda_tonic_slope': np.polyfit(np.arange(len(tonic)), tonic, 1)[0],
            'eda_phasic_mean': np.mean(phasic),
            'eda_phasic_std':  np.std(phasic),
            'eda_scr_count':   int(np.sum(peaks)),
        }
    except Exception:
        return {k: np.nan for k in [
            'eda_tonic_mean','eda_tonic_std','eda_tonic_slope',
            'eda_phasic_mean','eda_phasic_std','eda_scr_count'
        ]}


def resp_features(resp: np.ndarray, fs: int) -> dict:
    try:
        signals, _ = nk.rsp_process(resp, sampling_rate=fs)
        rate = signals['RSP_Rate'].values
        return {
            'resp_rate_mean':      np.mean(rate),
            'resp_rate_std':       np.std(rate),
            'resp_rate_min':       np.min(rate),
            'resp_rate_max':       np.max(rate),
            'resp_amplitude_mean': np.mean(signals['RSP_Amplitude'].values),
        }
    except Exception:
        return {k: np.nan for k in [
            'resp_rate_mean','resp_rate_std','resp_rate_min',
            'resp_rate_max','resp_amplitude_mean'
        ]}


In [5]:
def segment_subject(data: dict) -> pd.DataFrame:
    chest  = data['signal']['chest']
    labels = data['label']

    ecg  = chest['ECG'].flatten()
    eda  = chest['EDA'].flatten()
    resp = chest['Resp'].flatten()

    rows = []
    for start in range(0, len(labels) - WINDOW + 1, STEP):
        end   = start + WINDOW
        win_labels = labels[start:end]

        unique, counts = np.unique(win_labels, return_counts=True)
        majority = unique[np.argmax(counts)]

        if majority not in LABEL_MAP:
            continue                           # skip amusement/meditation/transient

        if np.max(counts) / WINDOW < 0.80:
            continue                           # skip impure windows

        feats = {}
        feats.update(ecg_features( ecg[start:end],  FS))
        feats.update(eda_features( eda[start:end],  FS))
        feats.update(resp_features(resp[start:end], FS))
        feats['label'] = LABEL_MAP[majority]
        rows.append(feats)

    return pd.DataFrame(rows)


In [6]:
def build_dataset() -> pd.DataFrame:
    frames = []
    for sid in SUBJECTS:
        print(f"Processing {sid}...", end=' ')
        try:
            df = segment_subject(load_subject(sid))
            df['subject'] = sid
            frames.append(df)
            print(f"{len(df)} windows  "
                  f"(stress={df['label'].sum()}, non-stress={(df['label']==0).sum()})")
        except Exception as e:
            print(f"FAILED: {e}")
    return pd.concat(frames, ignore_index=True)


In [7]:
def run_loso(df: pd.DataFrame):
    feat_cols = [c for c in df.columns if c not in ('label','subject')]
    X = df[feat_cols].values
    y = df['label'].values
    groups = df['subject'].values

    all_true, all_pred, all_prob = [], [], []

    for train_idx, test_idx in LeaveOneGroupOut().split(X, y, groups):
        subj = np.unique(groups[test_idx])[0]
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        # Drop NaN rows
        X_tr, y_tr = X_tr[~np.isnan(X_tr).any(1)], y_tr[~np.isnan(X_tr).any(1)]
        mask_te = ~np.isnan(X_te).any(1)
        X_te, y_te = X_te[mask_te], y_te[mask_te]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_te = scaler.transform(X_te)

        spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
        clf = xgb.XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=spw,
            eval_metric='logloss', random_state=42,
        )
        clf.fit(X_tr, y_tr)

        preds = clf.predict(X_te)
        probs = clf.predict_proba(X_te)[:, 1]
        auc   = roc_auc_score(y_te, probs)
        print(f"  [{subj}]  AUC: {auc:.3f}")

        all_true.extend(y_te); all_pred.extend(preds); all_prob.extend(probs)

    print("\n── Overall LOSO ──────────────────────────────────")
    print(classification_report(all_true, all_pred,
                                 target_names=['Non-stress','Stress']))
    print(f"ROC-AUC: {roc_auc_score(all_true, all_prob):.3f}")


In [8]:
def train_final(df: pd.DataFrame, out: str = "stress_model.pkl"):
    feat_cols = [c for c in df.columns if c not in ('label','subject')]
    X = df[feat_cols].values
    y = df['label'].values
    mask = ~np.isnan(X).any(1)
    X, y = X[mask], y[mask]

    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)

    spw = (y == 0).sum() / max((y == 1).sum(), 1)
    clf = xgb.XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=spw,
        eval_metric='logloss', random_state=42,
    )
    clf.fit(X_s, y)

    joblib.dump({'model': clf, 'scaler': scaler, 'features': feat_cols}, out)
    print(f"\nFinal model saved → {out}")


In [9]:
# Build features from all subjects
df = build_dataset()
df.to_csv("wesad_features.csv", index=False)
print(f"\nTotal: {len(df)} windows  |  {df['label'].mean():.1%} stress\n")

# Evaluate with LOSO cross-validation
run_loso(df)

# Train and save final model on all subjects
train_final(df)

Processing S2... 56 windows  (stress=19, non-stress=37)
Processing S3... 57 windows  (stress=20, non-stress=37)
Processing S4... 57 windows  (stress=20, non-stress=37)
Processing S5... 59 windows  (stress=20, non-stress=39)
Processing S6... 59 windows  (stress=21, non-stress=38)
Processing S7... 59 windows  (stress=20, non-stress=39)
Processing S8... 59 windows  (stress=21, non-stress=38)
Processing S9... 58 windows  (stress=20, non-stress=38)
Processing S10... 60 windows  (stress=22, non-stress=38)
Processing S11... 59 windows  (stress=21, non-stress=38)
Processing S13... 59 windows  (stress=21, non-stress=38)
Processing S14... 59 windows  (stress=21, non-stress=38)
Processing S15... 60 windows  (stress=22, non-stress=38)
Processing S16... 61 windows  (stress=22, non-stress=39)
Processing S17... 61 windows  (stress=23, non-stress=38)

Total: 883 windows  |  35.4% stress

  [S10]  AUC: 0.312
  [S11]  AUC: 1.000
  [S13]  AUC: 0.994
  [S14]  AUC: 1.000
  [S15]  AUC: 0.956
  [S16]  AUC: 1